In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import workCon
import os
from eval import get_model_from_run
from tqdm import tqdm
import ipdb
import traceback
import random
import math
import generate_dataset
from scipy.integrate import solve_ivp
import pickle
import plotly.graph_objects as go

In [ ]:
def load_data(data_path):
    with open(data_path, 'rb') as f:
        data = pickle.load(f)

    
    # move the data to specified device
    # data_on_device = tuple(
    #     item.to("cuda:0") if isinstance(item, torch.Tensor) else item for item in data
    # )
    # return data_on_device

    for i, item in enumerate(data):
        if isinstance(item, torch.Tensor):
            print(f"Tensor {i}: shape={item.shape}, dtype={item.dtype}, min={item.min()}, max={item.max()}, device={item.device}")
            if torch.isnan(item).any():
                print(f" - Contains NaNs")
    return data

In [ ]:
# # data_path = "/home/esmith/incontextlearning/transformer_control/dataset_pendulum/picklefolder"
# data_path = "/data/esmith/dataset_pendulum/picklefolder"
# cartpoles = np.random.randint(0, 50000, 10)
# data = []
# for i in tqdm(cartpoles):
#     try:
#         data.append(load_data(os.path.join(data_path, f"multipendulum_{i}.pkl")))
#     except Exception as e:
#         print(f"Error loading {i}: {e}")
#         continue

# print(len(data))

In [ ]:
# print(f"Len of data[0]: {len(data[0])}")
# print(f"Shape of data[0][0]: {data[0][0].shape}")

In [ ]:
# ## visualize theta and theta_dot phase plot

# def plot_theta_and_theta_dot(data):
#     fig = go.Figure()
#     for i in range(len(data)):
#         theta = torch.squeeze(data[i][0])[:, 1].cpu().numpy()
#         theta_dot = torch.squeeze(data[i][0])[:, 3].cpu().numpy()
#         fig.add_trace(go.Scatter(x=theta, y=theta_dot, mode='lines', name=f'Cartpole {i}'))
#     fig.update_layout(title='Theta vs Theta Dot Phase Plot',
#                       xaxis_title='Theta',
#                       yaxis_title='Theta Dot')
#     fig.show()
# plot_theta_and_theta_dot(data)

In [ ]:
# ## visualize control inputs
# def plot_control_inputs(data):
#     fig = go.Figure()
#     for i in range(len(data)):
#         control_input = torch.squeeze(data[i][1]).cpu().numpy()
#         fig.add_trace(go.Scatter(x=np.arange(len(control_input)), y=np.abs(control_input), mode='lines', name=f'Cartpole {i}'))
#     fig.update_layout(title='Control Inputs',
#                       xaxis_title='Time Step',
#                       yaxis_title='Control Input',
#                       yaxis_type='log')
#     fig.show()

# def plot_control_inputs_scaled(data):
#     fig = go.Figure()
#     fig2 = go.Figure()
#     for i in range(len(data)):
#         control_input = torch.squeeze(data[i][1]).cpu().numpy()
#         scaled_control_input = (control_input )/ (300) ### approximate min-max scaling
#         scaled_control_input2 = [control_input[i]/(1*0.95**i) for i in range(len(control_input))]

#         # import pdb; pdb.set_trace()

#         fig.add_trace(go.Scatter(x=np.arange(len(scaled_control_input)), y=scaled_control_input, mode='lines', name=f'Cartpole {i}'))
#         fig2.add_trace(go.Scatter(x=np.arange(len(scaled_control_input2)), y=scaled_control_input2, mode='lines', name=f'Cartpole {i}'))
#     fig.update_layout(title='Scaled Control Inputs (Min-Max Scaling)',
#                       xaxis_title='Time Step',
#                       yaxis_title='Scaled Control Input')
#     fig.show()
#     fig2.update_layout(title='Scaled Control Inputs (Time Scaling)',
#                       xaxis_title='Time Step',
#                       yaxis_title='Scaled Control Input')
#     fig2.show()

# plot_control_inputs(data)
# plot_control_inputs_scaled(data)

In [ ]:
# data_path_pends = "/data/esmith/Dataset_Pendulum_ICL/picklefolder/batch_1.pkl"
# data_path_pends = "/data/esmith/Dataset_LinearSystem_ICL/picklefolder/batch_1.pkl"
# data_path_pends = "/data/esmith/Dataset_Cartpole_ICL/picklefolder/batch_2.pkl"
# data_path_pends = "/data/esmith/Dataset_Cartpole_Better_ICL/picklefolder/batch_4.pkl"
# data_path_pends = "/data/esmith/Dataset_Acrobot_AIGymWithNoise2_ICL/picklefolder/batch_4.pkl"
# data_path_pends = "/data/esmith/Dataset_Acrobot_AIGymWithNoiseShorter_ICL/picklefolder/batch_1000.pkl"
data_path_pends = "/data/esmith/Dataset_Acrobot_AIGymWithNoiseShorter_ICL/picklefolder_test_outofdistr/batch_test_1.pkl"
pends = np.random.randint(0, 2000, 50)
# pends = np.arange(172)
# pends = np.arange(100)
# data_pend = []
# xs_pend, ys_pend, cartmasses_pend, polemasses_pend, polelength_pend = load_data(data_path_pends)
xs_pend, ys_pend, link_length1, link_length2, link_mass1, link_mass2, true_xs = load_data(data_path_pends)
# xs_pend_cos_theta = torch.cos(xs_pend[:, :, 2])
# xs_pend_sin_theta = torch.sin(xs_pend[:, :, 2])
# xs_pend_updated = torch.cat((xs_pend[:, :, :2], xs_pend_cos_theta.unsqueeze(2), xs_pend_sin_theta.unsqueeze(2), xs_pend[:, :, 3:]), dim=2)
# print(f"xs_pend_updated shape: {xs_pend_updated.shape}")

# goal_state = torch.tensor([0.0, 0.0, 1.0, 0.0, 0.0], device=xs_pend.device)
# filter out states that don't end up close to the goal state
# goal_threshold = 0.1
# mask = torch.norm(xs_pend_updated[:, -1, :5] - goal_state, dim=1) < goal_threshold
# final_window = xs_pend_updated[:, -40:, :5]
# cos_theta_thresh = 0.9
# sin_theta_thresh = 0.3
# theta_dot_thresh = 1.0

# is_upright = (torch.abs(final_window[:, :, 2]) > cos_theta_thresh) & \
#        (torch.abs(final_window[:, :, 3]) < sin_theta_thresh) & \
#        (torch.abs(final_window[:, :, 4]) < theta_dot_thresh)

# mask = is_upright.all(dim=1)
# print(f"mask shape: {mask.shape}")


# print(f"mask: {mask}")
# xs_pend_update_new = xs_pend_updated[mask]
# print(f"xs_pend_update_new shape: {xs_pend_update_new.shape}")
# xs_pend = xs_pend_update_new
# ys_pend = ys_pend[mask]

# xs_pend = xs_pend[mask]
print(f"xs_pend shape before filtering: {xs_pend.shape}")
print(f"ys_pend shape before filtering: {ys_pend.shape}")
xs_pend = xs_pend[pends]
# ys_pend = ys_pend[mask]
ys_pend = ys_pend[pends]
# cartmasses_pend = cartmasses_pend[mask]
# polemasses_pend = polemasses_pend[mask]
# polelength_pend = polelength_pend[mask]



# print(xs_pend.shape)
# xs_pend = xs_pend.permute(1,0,2)
# ys_pend = ys_pend.permute(1,0)
# xs_pend = [xs_pend[i] for i in pends]
# ys_pend = [ys_pend[i] for i in pends]

## see device data is on
# print(f"xs_pend[0]: {xs_pend[0]}")
# print(f"ys_pend[0]: {ys_pend[0]}")
# print(f"xs_pend[0].device: {xs_pend[0].device}")
# print(f"ys_pend[0].device: {ys_pend[0].device}")
# print(type(masses_pend))
print(f"xs_pend shape: {xs_pend.shape}")
print(f"ys_pend shape: {ys_pend.shape}")

In [ ]:
## plot xs_pend using matplotlib
# plt.figure(figsize=(10, 6))
# fig1, ax1 = plt.subplots(figsize=(10, 6))
# fig2, ax2 = plt.subplots(figsize=(10, 6))
count = 0
filtered_xs_pend = []
filtered_ys_pend = []
for i in range(xs_pend.shape[0]):
    if np.sum(ys_pend[i,-500:, 1].cpu().numpy()) > 200:
        # ax1.scatter(np.arange(xs_pend.shape[1]), np.cos(xs_pend[i, :, 0].cpu().numpy()), label=f'Pendulum {i} Theta')
        # ax2.scatter(np.arange(ys_pend.shape[1]), ys_pend[i, :, 1].cpu().numpy(), label=f'Pendulum {i} Theta')
        filtered_xs_pend.append(xs_pend[i])
        filtered_ys_pend.append(ys_pend[i])
        count += 1
print(f"Plotted {count} pendulums with significant control input.")
# ax1.set_xlabel('Time Step')
# ax1.set_ylabel('Theta (rad)')
# ax1.set_title('Pendulum Theta Over Time')
# # plt.legend()
# ax1.grid()
# ax2.set_xlabel('Time Step')
# ax2.set_ylabel('Control Input')
# ax2.set_title('Pendulum Control Input Over Time')
# ax2.grid()
# plt.show()


In [ ]:
# def extract_window_fast(xs, ys, window_size=800):
#     before = window_size - 300
    
#     cos_theta1s = torch.cos(xs[:, :, 0])
    
#     new_xs_list = []
#     new_ys_list = []
#     idxes = []

#     for i in range(xs.shape[0]):
#         cos_theta1 = cos_theta1s[i]
#         avg_rev = torch.cumsum(cos_theta1.flip(0), dim=0) / torch.arange(1, len(cos_theta1)+1, device=cos_theta1.device)
#         avg_cumsum_rev = avg_rev.flip(0)
#         indices = torch.where(avg_cumsum_rev < -0.96)[0]

#         if len(indices) > 0:
#             idx = indices[0].item()
#             idxes.append(idx)

#             if idx + window_size <= xs.shape[1]:
#                 start = max(0, idx - before)
#                 end = start + window_size
#                 new_xs_list.append(xs[i, start:end])
#                 new_ys_list.append(ys[i, start:end])
#             else:
#                 new_xs_list.append(xs[i, -window_size:])
#                 new_ys_list.append(ys[i, -window_size:])
#         # else: automatically skip trajectory

#     if len(new_xs_list) == 0:
#         return torch.empty(0, window_size, xs.shape[2], device=xs.device), \
#                torch.empty(0, window_size, ys.shape[2], device=ys.device), \
#                []

#     new_xs = torch.stack(new_xs_list)
#     new_ys = torch.stack(new_ys_list)

#     return new_xs, new_ys, idxes

def extract_window_fast(xs, ys, window_size=800):
    before = window_size - 350
    
    cos_theta1s = torch.cos(xs[:, :, 0])
    
    new_xs_list = []
    new_ys_list = []
    idxes = []
    batch_idx_list = []

    for i in range(xs.shape[0]):
        cos_theta1 = cos_theta1s[i]
        avg_rev = torch.cumsum(cos_theta1.flip(0), dim=0) / torch.arange(1, len(cos_theta1)+1, device=cos_theta1.device)
        avg_cumsum_rev = avg_rev.flip(0)
        indices = torch.where(avg_cumsum_rev < -0.96)[0]

        if len(indices) > 0:
            idx = indices[0].item()
            idxes.append(idx)
            batch_idx_list.append(i)  # keep track of original batch index

            if idx + window_size <= xs.shape[1]:
                start = max(0, idx - before)
                end = start + window_size
                new_xs_list.append(xs[i, start:end])
                new_ys_list.append(ys[i, start:end])
            else:
                new_xs_list.append(xs[i, -window_size:])
                new_ys_list.append(ys[i, -window_size:])
        # else: automatically skip trajectory

    if len(new_xs_list) == 0:
        return torch.empty(0, window_size, xs.shape[2], device=xs.device), \
               torch.empty(0, window_size, ys.shape[2], device=ys.device), \
               [], []

    new_xs = torch.stack(new_xs_list)
    new_ys = torch.stack(new_ys_list)

    return new_xs, new_ys, batch_idx_list, idxes

xs_pend_windowed, ys_pend_windowed, batch_idxes, idxes = extract_window_fast(torch.stack(filtered_xs_pend), torch.stack(filtered_ys_pend), window_size=600) 
# xs_pend_windowed, ys_pend_windowed, batch_idxes, idxes = extract_window_fast(xs_pend, ys_pend, window_size=800)
print(f"xs_pend_windowed shape: {xs_pend_windowed.shape}")
print(f"ys_pend_windowed shape: {ys_pend_windowed.shape}")
print(f"batch_idxes: {batch_idxes}")
print(f"idxes: {idxes}")

# plot
fig1, ax1 = plt.subplots(figsize=(10, 6))
fig2, ax2 = plt.subplots(figsize=(10, 6))
count = 0
for i in range(xs_pend_windowed.shape[0]):
    ax1.scatter(np.arange(xs_pend_windowed.shape[1]), np.cos(xs_pend_windowed[i, :, 0].cpu().numpy()), label=f'Pendulum {i} Theta')
    ax2.scatter(np.arange(ys_pend_windowed.shape[1]), ys_pend_windowed[i, :, 1].cpu().numpy(), label=f'Pendulum {i} Control Input')
    count += 1
print(f"Plotted {count} pendulums with significant control input.")
ax1.set_xlabel('Time Step')
ax1.set_ylabel('Theta (rad)')
ax1.set_title('Pendulum Theta Over Time')
# plt.legend()
ax1.grid()
ax2.set_xlabel('Time Step')
ax2.set_ylabel('Control Input')
ax2.set_title('Pendulum Control Input Over Time')
ax2.grid()
plt.show()


In [ ]:
# def extract_window_fast(xs, ys, window_size= 800):
#     before = window_size - 300
    
#     cos_theta1s = torch.cos(xs[:, :, 0])
#     new_xs = torch.zeros((xs.shape[0], window_size, xs.shape[2]), device=xs.device)
#     new_ys = torch.zeros((ys.shape[0], window_size, ys.shape[2]), device=ys.device)
#     idxes = []
#     for i in range(xs.shape[0]):
#         cos_theta1 = cos_theta1s[i]
#         avg_rev = torch.cumsum(cos_theta1.flip(0), dim=0)/ torch.arange(1, len(cos_theta1)+1, device=cos_theta1.device)
#         avg_cumsum_rev = avg_rev.flip(0)
#         indices = torch.where(avg_cumsum_rev < -0.96)[0]
#         if len(indices) > 0:
#             idx = indices[0].item()
#             idxes.append(idx)
#             if idx + window_size <= xs.shape[1]:
#                 new_xs[i] = xs[i, idx-before: idx + window_size - before] if idx >= before else xs[i, 0: window_size]
#                 new_ys[i] = ys[i, idx-before: idx + window_size - before] if idx >= before else ys[i, 0: window_size]
#             else:
#                 new_xs[i] = xs[i, -window_size:]
#                 new_ys[i] = ys[i, -window_size:]
#         else:
#             pass
#     return new_xs, new_ys, idxes


# ## example usage
# new_xs, new_ys, idxes = extract_window_fast(torch.stack(filtered_xs_pend), torch.stack(filtered_ys_pend), window_size=500)
# print(f"new_xs shape: {new_xs.shape}")
# print(f"new_ys shape: {new_ys.shape}")
# print(f"idxes: {idxes}")
# fig3, ax3 = plt.subplots(figsize=(10, 6))
# fig4, ax4 = plt.subplots(figsize=(10, 6))
# count = 0
# for i in range(new_xs.shape[0]):
#     # if np.sum(ys[i,:, 1].cpu().numpy()) > 200:
#     ax3.scatter(np.arange(new_xs.shape[1]), np.cos(new_xs[i, :, 0].cpu().numpy()), label=f'Pendulum {i} Theta')
#     ax4.scatter(np.arange(new_ys.shape[1]), new_ys[i, :, 1].cpu().numpy(), label=f'Pendulum {i} Theta')
#     count += 1
# print(f"Plotted {count} pendulums with significant control input.")
# ax3.set_xlabel('Time Step')
# ax3.set_ylabel('Theta (rad)')
# ax3.set_title('Pendulum Theta Over Time (Windowed)')
# ax3.grid()
# ax4.set_xlabel('Time Step')
# ax4.set_ylabel('Control Input')
# ax4.set_title('Pendulum Control Input Over Time (Windowed)')
# ax4.grid()
# plt.show()


In [ ]:
### plot pendulum data
testing_traj_x1= []
testing_traj_x2 = []
def plot_pendulum_data(xs):
    fig = go.Figure()
    fig2 = go.Figure()
    fig3 = go.Figure()
    fig4 = go.Figure()
    fig5 = go.Figure()
    for i in range(len(xs_pend)):
        # x = np.cos(xs[i][:, 0].cpu().numpy()) #theta
        # x = np.where(x > 1e-3, x, 20)
        # mask = np.abs(x) > 1e-3
        # x = x[mask]
        # x = np.cos(x)
        # y = np.cos(xs[i][:, 1].cpu().numpy()) #theta_dot

        x = xs[i][:, 2].cpu().numpy() #theta1dot
        y = xs[i][:, 3].cpu().numpy() #theta2dot

        

        # adding decaying noise to the data
        # decay = np.linspace(1, 0.01, len(x))
        # stdv = 0.01
        # noise_x = np.random.randn(len(x)) * decay * stdv
        # noise_y = np.random.randn(len(y)) * decay * stdv

        # x = x + noise_x
        # y = y + noise_y
        
        testing_traj_x1.append(x)
        testing_traj_x2.append(y)
        # fig.add_trace(go.Scatter(x=x, y=y, mode='lines', name=f'Pendulum {i}'))
        fig.add_trace(go.Scatter(x=np.arange(len(x)), y=x, mode='lines', name=f'System {i}'))
        fig2.add_trace(go.Scatter(x=np.arange(len(x)), y=np.abs(x), mode='lines', name=f'System {i}'))
        fig3.add_trace(go.Scatter(x=np.arange(len(y)), y=y, mode='lines', name=f'System {i}'))
        fig4.add_trace(go.Scatter(x=np.arange(len(y)), y=np.abs(y), mode='lines', name=f'System {i}'))
        fig5.add_trace(go.Scatter(x=x, y=y, mode='lines', name=f'System {i}'))
        # fig.add_trace(go.Scatter(x=np.arange))
    fig.update_layout(title='Time Series theta',
                      xaxis_title='Time',
                      yaxis_title='theta')
    fig2.update_layout(title='Time Series theta(Abs)',
                      xaxis_title='Time',
                      yaxis_title='theta(Abs)',
                      yaxis_type='log')
    fig3.update_layout(title='Time Series thetadot',
                      xaxis_title='Time',
                      yaxis_title='thetadot')
    fig4.update_layout(title='Time Series thetadot (Abs)',
                      xaxis_title='Time',
                      yaxis_title='thetadot (Abs)',
                      yaxis_type='log')
    # fig5.update_layout(title='Pendulum Phase Plot',
    #                   xaxis_title='Theta',
    #                   yaxis_title='Theta Dot')

    # fig5.show()
    fig.show()
    fig2.show()
    fig3.show()
    fig4.show()
    # return

def plot_pendulum_data_scaled(xs):
    fig = go.Figure()
    fig2 = go.Figure()
    fig3 = go.Figure()
    fig4 = go.Figure()
    for i in range(len(xs_pend)):
        x = xs[i][:, 0].cpu().numpy() #theta
        y = xs[i][:, 1].cpu().numpy() #theta_dot
        # mask = np.abs(x) > 1e-3
        # x = x[mask]
        # scaled_x = (x - np.min(x)) / (np.max(x) - np.min(x))
        # scaled_y = (y - np.min(y)) / (np.max(y) - np.min(y))
        # scaled_x = [x[i] / (40 * 0.98 ** i) for i in range(len(x))]
        # scaled_y = [y[i] / (40 * 0.98 ** i) for i in range(len(y))]
        
        # scaled_x = np.sign(x) * np.log1p(1e5*np.abs(x))/10
        # scaled_y = np.sign(y) * np.log1p(1e5*np.abs(y))/15
        # a_theta = -0.0288 #-0.0274
        # b_theta = 3.5833 #1.0035

        # a_thetadot = -0.0288 #-0.0259
        # b_thetadot = 3.5833 #1.7668

        # a = -0.03869
        # b = 2.15125

        a = -0.036096496474688156
        b = 3.292495439274396

        a_theta = a
        b_theta = b

        a_thetadot = a
        b_thetadot = b

        # scaled_x = scaled_y = np.sign(y)*(np.abs(y)/(np.exp(a*np.arange(len(y))+b) + 1e-12))/75
        scaled_x = np.sign(x)*(np.abs(x)/(np.exp(a_theta*np.arange(len(x))+b_theta) + 1e-12)) #/150
        scaled_y = np.sign(y)*(np.abs(y)/(np.exp(a_thetadot*np.arange(len(y))+b_thetadot) + 1e-12)) #/150 #50

        # adding decaying noise to the data
        # decay = np.linspace(1, 0.01, len(scaled_x))
        # stdv = 0.01
        # noise_x = np.random.randn(len(scaled_x)) * decay * stdv
        # noise_y = np.random.randn(len(scaled_y)) * decay * stdv

        # scaled_x = scaled_x + noise_x
        # scaled_y = scaled_y + noise_y
        

        # first_fifty_indices = np.arange(0, 50)
        # last_threefifty_indices = np.arange(50, 400)
        # scaled_x = np.zeros_like(x)
        # scaled_y = np.zeros_like(y)
        # scaled_x[first_fifty_indices] = x[first_fifty_indices]/5
        # scaled_y[first_fifty_indices] = y[first_fifty_indices]/15

        # scaled_x[last_threefifty_indices] = x[last_threefifty_indices]/(5*0.98**(last_threefifty_indices-50))
        # scaled_y[last_threefifty_indices] = y[last_threefifty_indices]/(5*0.98**(last_threefifty_indices-50))
        # scaled_x = np.arcsinh(x/1e-3)/8
        # scaled_y = np.arcsinh(y/1e-3)/8
        # fig.add_trace(go.Scatter(x=scaled_x, y=scaled_y, mode='markers', name=f'Pendulum {i}'))
        fig.add_trace(go.Scatter(x=np.arange(len(scaled_x)), y=scaled_x, mode='lines', name=f'System {i}'))
        fig2.add_trace(go.Scatter(x=np.arange(len(scaled_x)), y=np.abs(scaled_x), mode='lines', name=f'System {i}'))
        fig3.add_trace(go.Scatter(x=np.arange(len(scaled_y)), y=scaled_y, mode='lines', name=f'System {i}'))
        fig4.add_trace(go.Scatter(x=np.arange(len(scaled_y)), y=np.abs(scaled_y), mode='lines', name=f'System {i}'))
    fig.update_layout(title='Scaled x1',
                      xaxis_title='Time',
                      yaxis_title='Scaled x1'
                      )
    fig2.update_layout(title='Scaled x1 (Abs)',
                      xaxis_title='Time',
                      yaxis_title='Scaled x1 (Abs)',
                      yaxis_type='log'
                      )
    fig3.update_layout(title='Scaled x2',
                      xaxis_title='Time',
                      yaxis_title='Scaled x2'
                      )
    fig4.update_layout(title='Scaled x2 (Abs)',
                      xaxis_title='Time',
                      yaxis_title='Scaled x2 (Abs)',
                      yaxis_type='log'
                      )
    
    
    fig.show()
    fig2.show()
    fig3.show()
    fig4.show()


# plot_pendulum_data(xs_pend)
plot_pendulum_data(true_xs)
print("--" * 50)
# plot_pendulum_data_scaled(xs_pend)